# Neural Network From Scratch: Solve XOR

*1991 Academy · The Lab*

XOR is not linearly separable — no single neuron can solve it (that discovery froze neural-net research for a decade). A network with one hidden layer can. You're going to build it: forward pass, backpropagation, updates. No libraries.

**Architecture:** 2 inputs → 4 hidden (tanh) → 1 output (sigmoid), squared-error loss, per-sample SGD.

**Forward**, for input (x1, x2):

```
h[j] = tanh(W1[j][0]·x1 + W1[j][1]·x2 + b1[j])     j = 0..3
z    = Σ W2[j]·h[j] + b2
p    = 1 / (1 + e^(−z))          # the prediction
```

**Backward** (chain rule, exactly as in the backprop lesson):

```
dz     = 2·(p − y) · p·(1 − p)
dW2[j] = dz · h[j]        db2 = dz
dh[j]  = dz · W2[j]
dpre[j]= dh[j] · (1 − h[j]²)      # tanh derivative
dW1[j][0] = dpre[j]·x1    dW1[j][1] = dpre[j]·x2    db1[j] = dpre[j]
```

Update every parameter: `param −= lr · grad` after each sample. Record the average loss over the 4 XOR samples once per epoch. Return `{"predict": predict, "loss_history": loss_history}`, where `predict(x1, x2)` runs your forward pass.

The notebook paints your network's output over the whole input square, plus your loss curve. Watching the four corners separate is watching a hidden layer *fold space*.

> 💾 **Your work is saved:** uploaded to Colab (**File → Upload notebook**), this notebook lives in your Google Drive, in the *Colab Notebooks* folder. Close it any time and continue later.
>
> ✅ When every test passes, go back to 1991 Academy and click **I solved it ✓**.

## Your code

Write your solution in the cell below and run it: click ▶, or press Shift+Enter.

In [ ]:
# Build and train a 2-4-1 network on XOR.
# Return {"predict": predict, "loss_history": loss_history}.
def train_xor():
    X = [[0, 0], [0, 1], [1, 0], [1, 1]]
    Y = [0, 1, 1, 0]
    # fixed starting weights: everyone converges the same way
    W1 = [[0.5, -0.4], [0.3, 0.8], [-0.6, 0.2], [0.7, -0.3]]
    b1 = [0.1, -0.2, 0.05, 0.15]
    W2 = [0.4, -0.5, 0.6, 0.3]
    b2 = 0.05
    lr, epochs = 0.5, 4000
    loss_history = []

    def forward(x1, x2):
        # return (h, p): 4 hidden tanh activations and the sigmoid output
        pass

    for e in range(epochs):
        loss = 0.0
        for s in range(4):
            # forward, accumulate (p - y)**2 into loss,
            # backward (formulas in the brief), update all params
            pass
        loss_history.append(loss / 4)

    def predict(x1, x2):
        return forward(x1, x2)[1]

    return {"predict": predict, "loss_history": loss_history}

## Check your solution

Run the cell below. If a test fails, fix your code, run your code cell again, then this one.

In [ ]:
#@title ▶ Run the tests { display-mode: "form" }
import json, math, traceback
MSG = {
 "all_pass": "🎉 All {n} tests pass! Go back to 1991 Academy and click “I solved it ✓”.",
 "some_pass": "{p}/{n} tests pass. Keep going!",
 "none_ran": "No tests ran. Is your function named {fn}?",
 "expected": "expected {e}, got {a}",
 "error": "💥 The tests stopped with an error:",
 "not_run": "Did you run your code cell above? The tests only see code that has been run."
}

def _report(results, error, fn):
    for name, ok, expected, actual in results:
        print(("✓ " if ok else "✗ ") + name + ("" if ok else "   " + MSG["expected"].format(e=expected, a=actual)))
    if error:
        print("\n" + MSG["error"])
        print(error.rstrip())
    passed = sum(1 for r in results if r[1])
    print()
    if results and passed == len(results) and not error:
        print(MSG["all_pass"].format(n=passed))
    elif results:
        print(MSG["some_pass"].format(p=passed, n=len(results)))
    elif not error:
        print(MSG["none_ran"].format(fn=fn))

def _academy_run_tests():
    results = []

    def __check(name, actual, expected):
        try:
            ok = bool(actual == expected)
        except Exception:
            ok = False
        results.append((str(name), ok, repr(expected), repr(actual)))

    def tests():
        net = None
        for attempt in range(3):
            n = train_xor()
            if n and callable(n.get("predict")):
                ok = (n["predict"](0, 0) < 0.3 and n["predict"](1, 1) < 0.3 and
                      n["predict"](0, 1) > 0.7 and n["predict"](1, 0) > 0.7)
                if ok or attempt == 2:
                    net = n
                    break
        __check("returns predict + loss_history", bool(net and callable(net.get("predict")) and isinstance(net.get("loss_history"), list)), True)
        __check("(0,0) -> ~0", net["predict"](0, 0) < 0.3, True)
        __check("(1,1) -> ~0", net["predict"](1, 1) < 0.3, True)
        __check("(0,1) -> ~1", net["predict"](0, 1) > 0.7, True)
        __check("(1,0) -> ~1", net["predict"](1, 0) > 0.7, True)
        __check("loss went DOWN during training", net["loss_history"][0] > net["loss_history"][-1] * 3, True)

    error = None
    try:
        tests()
    except Exception as exc:
        error = traceback.format_exc(limit=-3)
        if isinstance(exc, NameError):
            error += "\n" + MSG["not_run"]
    _report(results, error, 'train_xor')

_academy_run_tests()

## Hints

<details><summary>Hint 1</summary>

Write forward() first and test it mentally: with the given starting weights, predict(0,0) should return some number near 0.5 before training.

</details>

<details><summary>Hint 2</summary>

Backprop order matters: compute dz first, then dW2/db2, then dh -> dpre -> dW1/db1 — and only THEN apply all the updates for this sample.

</details>

## See your code run

Once the tests pass, run this cell to see what your code does.

In [ ]:
#@title 📊 Visualize your code { display-mode: "form" }
import json
import matplotlib.pyplot as plt
VIZ = json.loads(r"""{}""")
net = train_xor()
cells, lo, hi = 56, -0.25, 1.25
heat = [[net["predict"](lo + (c + 0.5) / cells * (hi - lo), hi - (r + 0.5) / cells * (hi - lo)) for c in range(cells)]
        for r in range(cells)]
fig, (left, right) = plt.subplots(1, 2, figsize=(10, 4.2))
left.imshow(heat, extent=(lo, hi, lo, hi), cmap="coolwarm", vmin=0, vmax=1)
left.scatter([0, 1, 0, 1], [0, 1, 1, 0], c=[0, 0, 1, 1], cmap="coolwarm", edgecolors="black", s=120, vmin=0, vmax=1)
left.set_title("your network's output over the input square")
right.plot(net["loss_history"], color="#8b5cf6")
right.set_title("loss per epoch"); right.set_xlabel("epoch")
plt.tight_layout(); plt.show()